# AutoDL 实验运行 Notebook

在 AutoDL JupyterLab 里逐步运行所有实验。

> **使用前提**：已通过 scp / JupyterLab 上传 `data/billsum/` 和 `data/casehold/`

## 0. 环境准备

In [ ]:
import os
# 确认在正确目录
os.chdir('/root/MLP')
!pwd
!ls

In [ ]:
# 安装依赖（第一次运行，约 5 分钟）
# AutoDL 学术加速（加快 pip 下载速度）
!source /etc/network_turbo 2>/dev/null || true
!bash scripts/setup_env.sh

In [ ]:
# 验证环境
!conda run -n llm-ft python -c "import torch, peft, trl, bitsandbytes; print('torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"

In [ ]:
# 验证数据文件
import os
files = [
    'data/billsum/train_sft.jsonl',
    'data/billsum/val_sft.jsonl',
    'data/billsum/test_us_sft.jsonl',
    'data/billsum/test_ca_sft.jsonl',
    'data/casehold/train_mc.jsonl',
    'data/casehold/validation_mc.jsonl',
    'data/casehold/test_mc.jsonl',
]
for f in files:
    size = os.path.getsize(f) // 1024 if os.path.exists(f) else -1
    status = f'✓ {size} KB' if size >= 0 else '✗ 缺失！'
    print(f'{status}  {f}')

## 1. BillSum LoRA — Qwen

In [ ]:
%%bash
cd /root/MLP
conda run -n llm-ft python src/train/train.py --config configs/lora_billsum_qwen.yaml 2>&1 | tee logs/lora_billsum_qwen.log
echo "训练完成: $(date)"

In [ ]:
%%bash
cd /root/MLP
conda run -n llm-ft python src/evaluate/inference.py --config configs/lora_billsum_qwen.yaml --split test_us
conda run -n llm-ft python src/evaluate/inference.py --config configs/lora_billsum_qwen.yaml --split test_ca
conda run -n llm-ft python src/evaluate/eval_billsum.py \
    --predictions outputs/lora_billsum_qwen/predictions_test_us.jsonl \
    --output outputs/lora_billsum_qwen/eval_test_us.json
conda run -n llm-ft python src/evaluate/eval_billsum.py \
    --predictions outputs/lora_billsum_qwen/predictions_test_ca.jsonl \
    --output outputs/lora_billsum_qwen/eval_test_ca.json
echo "评估完成"
cat outputs/lora_billsum_qwen/eval_test_us.json

## 2. BillSum LoRA — Llama

In [ ]:
# Llama 需要先登录 HuggingFace（有 license 的话）
# !conda run -n llm-ft huggingface-cli login
# 如果没有 Llama 访问权，跳过这个 cell，直接跑 Full FT

In [ ]:
%%bash
cd /root/MLP
conda run -n llm-ft python src/train/train.py --config configs/lora_billsum_llama.yaml 2>&1 | tee logs/lora_billsum_llama.log
conda run -n llm-ft python src/evaluate/inference.py --config configs/lora_billsum_llama.yaml --split test_us
conda run -n llm-ft python src/evaluate/inference.py --config configs/lora_billsum_llama.yaml --split test_ca
conda run -n llm-ft python src/evaluate/eval_billsum.py \
    --predictions outputs/lora_billsum_llama/predictions_test_us.jsonl \
    --output outputs/lora_billsum_llama/eval_test_us.json
conda run -n llm-ft python src/evaluate/eval_billsum.py \
    --predictions outputs/lora_billsum_llama/predictions_test_ca.jsonl \
    --output outputs/lora_billsum_llama/eval_test_ca.json
cat outputs/lora_billsum_llama/eval_test_us.json

## 3. BillSum Full FT — Qwen

In [ ]:
%%bash
cd /root/MLP
conda run -n llm-ft python src/train/train.py --config configs/full_billsum_qwen.yaml 2>&1 | tee logs/full_billsum_qwen.log
conda run -n llm-ft python src/evaluate/inference.py --config configs/full_billsum_qwen.yaml --split test_us
conda run -n llm-ft python src/evaluate/inference.py --config configs/full_billsum_qwen.yaml --split test_ca
conda run -n llm-ft python src/evaluate/eval_billsum.py \
    --predictions outputs/full_billsum_qwen/predictions_test_us.jsonl \
    --output outputs/full_billsum_qwen/eval_test_us.json
conda run -n llm-ft python src/evaluate/eval_billsum.py \
    --predictions outputs/full_billsum_qwen/predictions_test_ca.jsonl \
    --output outputs/full_billsum_qwen/eval_test_ca.json
cat outputs/full_billsum_qwen/eval_test_us.json

## 4. BillSum Full FT — Llama

In [ ]:
%%bash
cd /root/MLP
conda run -n llm-ft python src/train/train.py --config configs/full_billsum_llama.yaml 2>&1 | tee logs/full_billsum_llama.log
conda run -n llm-ft python src/evaluate/inference.py --config configs/full_billsum_llama.yaml --split test_us
conda run -n llm-ft python src/evaluate/inference.py --config configs/full_billsum_llama.yaml --split test_ca
conda run -n llm-ft python src/evaluate/eval_billsum.py \
    --predictions outputs/full_billsum_llama/predictions_test_us.jsonl \
    --output outputs/full_billsum_llama/eval_test_us.json
conda run -n llm-ft python src/evaluate/eval_billsum.py \
    --predictions outputs/full_billsum_llama/predictions_test_ca.jsonl \
    --output outputs/full_billsum_llama/eval_test_ca.json
cat outputs/full_billsum_llama/eval_test_us.json

## 5. CaseHOLD LoRA — Qwen

In [ ]:
%%bash
cd /root/MLP
conda run -n llm-ft python src/train/train.py --config configs/lora_casehold_qwen.yaml 2>&1 | tee logs/lora_casehold_qwen.log
conda run -n llm-ft python src/evaluate/inference.py --config configs/lora_casehold_qwen.yaml --split test
conda run -n llm-ft python src/evaluate/eval_casehold.py \
    --predictions outputs/lora_casehold_qwen/predictions_test.jsonl \
    --output outputs/lora_casehold_qwen/eval_test.json
cat outputs/lora_casehold_qwen/eval_test.json

## 6. CaseHOLD LoRA — Llama

In [ ]:
%%bash
cd /root/MLP
conda run -n llm-ft python src/train/train.py --config configs/lora_casehold_llama.yaml 2>&1 | tee logs/lora_casehold_llama.log
conda run -n llm-ft python src/evaluate/inference.py --config configs/lora_casehold_llama.yaml --split test
conda run -n llm-ft python src/evaluate/eval_casehold.py \
    --predictions outputs/lora_casehold_llama/predictions_test.jsonl \
    --output outputs/lora_casehold_llama/eval_test.json
cat outputs/lora_casehold_llama/eval_test.json

## 7. CaseHOLD QLoRA — Qwen

In [ ]:
%%bash
cd /root/MLP
conda run -n llm-ft python src/train/train_casehold_lora.py --config configs/qlora_casehold_qwen.yaml 2>&1 | tee logs/qlora_casehold_qwen.log
conda run -n llm-ft python src/evaluate/inference.py --config configs/qlora_casehold_qwen.yaml --split test
conda run -n llm-ft python src/evaluate/eval_casehold.py \
    --predictions outputs/qlora_casehold_qwen/predictions_test.jsonl \
    --output outputs/qlora_casehold_qwen/eval_test.json
cat outputs/qlora_casehold_qwen/eval_test.json

## 8. CaseHOLD QLoRA — Llama

In [ ]:
%%bash
cd /root/MLP
conda run -n llm-ft python src/train/train_casehold_lora.py --config configs/qlora_casehold_llama.yaml 2>&1 | tee logs/qlora_casehold_llama.log
conda run -n llm-ft python src/evaluate/inference.py --config configs/qlora_casehold_llama.yaml --split test
conda run -n llm-ft python src/evaluate/eval_casehold.py \
    --predictions outputs/qlora_casehold_llama/predictions_test.jsonl \
    --output outputs/qlora_casehold_llama/eval_test.json
cat outputs/qlora_casehold_llama/eval_test.json

## 汇总所有结果

In [ ]:
import json, os

results = [
    ('lora_billsum_qwen',   'outputs/lora_billsum_qwen/eval_test_us.json'),
    ('lora_billsum_llama',  'outputs/lora_billsum_llama/eval_test_us.json'),
    ('full_billsum_qwen',   'outputs/full_billsum_qwen/eval_test_us.json'),
    ('full_billsum_llama',  'outputs/full_billsum_llama/eval_test_us.json'),
    ('lora_casehold_qwen',  'outputs/lora_casehold_qwen/eval_test.json'),
    ('lora_casehold_llama', 'outputs/lora_casehold_llama/eval_test.json'),
    ('qlora_casehold_qwen', 'outputs/qlora_casehold_qwen/eval_test.json'),
    ('qlora_casehold_llama','outputs/qlora_casehold_llama/eval_test.json'),
]

print(f'{"实验":<25} {"指标":<15} {"值":>8}')
print('-' * 52)
for name, path in results:
    if not os.path.exists(path):
        print(f'{name:<25} {"未完成":<15}')
        continue
    with open(path) as f:
        d = json.load(f)
    # BillSum 看 rouge2，CaseHOLD 看 accuracy
    if 'rouge2' in d:
        print(f'{name:<25} {"rouge2":<15} {d["rouge2"]:>8.4f}')
    elif 'accuracy' in d:
        print(f'{name:<25} {"accuracy":<15} {d["accuracy"]:>8.4f}')